In [3]:
import re
import unicodedata
from collections import Counter

import pandas as pd

In [ ]:
data = pd.read_csv("arabic_categorization_data.csv")

print(data.shape)
print(data.head(3))

In [ ]:
required_cols = ["text", "type"]

if "id" not in data.columns:
    if "Unnamed: 0" in data.columns:
        data = data.rename(columns={"Unnamed: 0": "id"})
    else:
        data["id"] = range(len(data))

data = data[["id", "type", "text"]].copy()

In [ ]:
data["id"] = data["id"]
data["type"] = data["type"].astype(str)
data["text"] = data["text"].fillna("").astype(str)

print("Missing values:")
print(data.isna().sum())

print("\nEmpty text rows:", (data["text"].str.strip() == "").sum())
print("Duplicate full rows:", data.duplicated().sum())
print("Duplicate text rows:", data.duplicated(subset=["text"]).sum())

print("\nType distribution:")
print(data["type"].value_counts(dropna=False))

In [ ]:
ARABIC_RE = re.compile(r'[\u0600-\u06FF]')
ENGLISH_RE = re.compile(r'[A-Za-z]')
HTML_TAG_RE = re.compile(r'<[^>]+>')
URL_RE = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')
DIACRITICS_RE = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670]')
TATWEEL_RE = re.compile(r'ـ')
REPEATED_PUNCT_RE = re.compile(r'([!?؟،؛.,:])\1{1,}')
DIGIT_RE = re.compile(r'\d')
ARABIC_VARIANTS_RE = re.compile(r'[إأآٱ]')
CONTROL_CHARS_RE = re.compile(r'[\x00-\x1F\x7F]')
INVISIBLE_UNICODE_RE = re.compile(r'[\u200b-\u200f\u202a-\u202e\u2066-\u2069]')

In [ ]:
def inspect_text(text):
    text = "" if pd.isna(text) else str(text)

    total_chars = len(text)
    word_count = len(text.split())

    arabic_chars = len(ARABIC_RE.findall(text))
    english_chars = len(ENGLISH_RE.findall(text))
    digit_chars = len(DIGIT_RE.findall(text))

    has_html = bool(HTML_TAG_RE.search(text))
    has_url = bool(URL_RE.search(text))
    has_email = bool(EMAIL_RE.search(text))
    has_diacritics = bool(DIACRITICS_RE.search(text))
    has_tatweel = bool(TATWEEL_RE.search(text))
    has_repeated_punct = bool(REPEATED_PUNCT_RE.search(text))
    has_variants = bool(ARABIC_VARIANTS_RE.search(text))
    has_control_chars = bool(CONTROL_CHARS_RE.search(text))
    has_invisible_unicode = bool(INVISIBLE_UNICODE_RE.search(text))

    suspicious_chars = []
    for ch in text:
        if ch in "\n\r\t ":
            continue
        if ARABIC_RE.match(ch) or ENGLISH_RE.match(ch) or ch.isdigit():
            continue
        if ch in ".,!?؟،؛:()[]{}\"'«»-_/\\%+*=|<>~":
            continue

        cat = unicodedata.category(ch)
        if cat.startswith("C") or cat.startswith("S"):
            suspicious_chars.append(ch)

    arabic_ratio = arabic_chars / total_chars if total_chars > 0 else 0.0
    english_ratio = english_chars / total_chars if total_chars > 0 else 0.0

    return {
        "char_len": total_chars,
        "word_count": word_count,
        "arabic_chars": arabic_chars,
        "english_chars": english_chars,
        "digit_chars": digit_chars,
        "arabic_ratio": arabic_ratio,
        "english_ratio": english_ratio,
        "has_html": has_html,
        "has_url": has_url,
        "has_email": has_email,
        "has_diacritics": has_diacritics,
        "has_tatweel": has_tatweel,
        "has_repeated_punct": has_repeated_punct,
        "has_variants": has_variants,
        "has_control_chars": has_control_chars,
        "has_invisible_unicode": has_invisible_unicode,
        "suspicious_char_count": len(suspicious_chars),
        "suspicious_chars_sample": "".join(sorted(set(suspicious_chars)))[:100],
    }

diag = data["text"].apply(inspect_text).apply(pd.Series)
scan_df = pd.concat([data, diag], axis=1)

print("DIAGNOSTIC SUMMARY")
summary = {
    "rows": len(scan_df),
    "empty_text_rows": (scan_df["text"].str.strip() == "").sum(),
    "duplicate_text_rows": scan_df.duplicated(subset=["text"]).sum(),
    "rows_with_html": scan_df["has_html"].sum(),
    "rows_with_urls": scan_df["has_url"].sum(),
    "rows_with_emails": scan_df["has_email"].sum(),
    "rows_with_diacritics": scan_df["has_diacritics"].sum(),
    "rows_with_tatweel": scan_df["has_tatweel"].sum(),
    "rows_with_repeated_punct": scan_df["has_repeated_punct"].sum(),
    "rows_with_arabic_variants": scan_df["has_variants"].sum(),
    "rows_with_english": (scan_df["english_chars"] > 0).sum(),
    "rows_with_digits": (scan_df["digit_chars"] > 0).sum(),
    "rows_with_suspicious_symbols": (scan_df["suspicious_char_count"] > 0).sum(),
    "rows_with_invisible_unicode": scan_df["has_invisible_unicode"].sum(),
    "rows_with_low_arabic_ratio_<0.6": (scan_df["arabic_ratio"] < 0.6).sum(),
    "very_short_rows_<5_words": (scan_df["word_count"] < 5).sum(),
}

for k, v in summary.items():
    print(f"{k}: {v}")

print("Len Stats")
print(scan_df["word_count"].describe(percentiles=[0.50, 0.90, 0.95, 0.99]))


In [ ]:
def show_examples(df, condition, n=3, text_col="text", label="Examples", max_chars=1500):
    sample = df.loc[condition, ["id", "type", text_col]].head(n)
    print(f"{label}")
    if sample.empty:
        print("No examples found.")
        return
    for _, row in sample.iterrows():
        print(f"\nID: {row['id']} | Type: {row['type']}")
        print(str(row[text_col])[:max_chars])
        print("-" * 80)

show_examples(scan_df, scan_df["has_html"], label="Rows with HTML")
show_examples(scan_df, scan_df["has_url"], label="Rows with URLs")
show_examples(scan_df, scan_df["english_chars"] > 0, label="Rows with English")
show_examples(scan_df, scan_df["suspicious_char_count"] > 0, label="Rows with suspicious symbols")
show_examples(scan_df, scan_df["has_diacritics"], label="Rows with diacritics")
show_examples(scan_df, scan_df["has_tatweel"], label="Rows with tatweel")
show_examples(scan_df, scan_df["has_invisible_unicode"], label="Rows with invisible unicode")

In [ ]:
ARABIC_DIACRITICS_FULL = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670]|ـ')

def remove_html(text):
    return re.sub(r'<[^>]+>', ' ', text)

def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', ' ', text)

def remove_emails(text):
    return re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' ', text)

def remove_control_chars(text):
    text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)
    text = re.sub(r'[\u200b-\u200f\u202a-\u202e\u2066-\u2069]', ' ', text)
    return text

def remove_diacritics(text):
    return re.sub(ARABIC_DIACRITICS_FULL, '', text)

def normalize_arabic(text):
    # light normalization only
    text = re.sub(r'[إأآٱ]', 'ا', text)
    # keep ى as-is
    # keep ة as-is
    # keep ؤ and ئ as-is
    return text

def normalize_repeated_punctuation(text):
    return re.sub(r'([!?؟،؛.,:])\1{1,}', r'\1', text)

def clean_noise(text):
    # keep Arabic, English, digits, whitespace, and useful punctuation
    # keep numbers and English because they may carry meaning in news text
    text = re.sub(r'[^\u0600-\u06FFA-Za-z0-9\s\.\,\!\?\؟\،\؛\:\-\(\)\"\'/٪%+]', ' ', text)
    return text

def collapse_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_text(text):
    text = "" if pd.isna(text) else str(text)

    text = remove_html(text)
    text = remove_urls(text)
    text = remove_emails(text)
    text = remove_control_chars(text)
    text = remove_diacritics(text)
    text = normalize_arabic(text)
    text = normalize_repeated_punctuation(text)
    text = clean_noise(text)
    text = collapse_spaces(text)

    return text

In [ ]:
scan_df["clean_text"] = scan_df["text"].apply(preprocess_text)

scan_df["clean_word_count"] = scan_df["clean_text"].apply(lambda x: len(str(x).split()))

print("Empty clean_text rows:", (scan_df["clean_text"].str.strip() == "").sum())
print("Duplicate clean_text rows:", scan_df.duplicated(subset=["clean_text"]).sum())

print("\nClean text length stats:")
print(scan_df["clean_word_count"].describe(percentiles=[0.50, 0.90, 0.95, 0.99]))

In [ ]:
processed_df = scan_df.copy()

# remove empty cleaned rows
processed_df = processed_df[processed_df["clean_text"].str.strip() != ""]

# remove very short rows
processed_df = processed_df[processed_df["clean_word_count"] >= 5]

# remove rows with very low Arabic content
processed_df = processed_df[processed_df["arabic_ratio"] >= 0.6]

# remove duplicates after cleaning
processed_df = processed_df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

print("\nProcessed shape after filtering:", processed_df.shape)


In [ ]:
processed_df[
    ["id", "type", "text", "clean_text", "word_count", "clean_word_count", "arabic_ratio"]
].to_csv(
    "arabic_articles_preprocessed.csv",
    index=False,
    encoding="utf-8-sig"
)